# Preprocessing: Polar 6 GPS/INS Master Tracks

**Manuscript reference:** Sect. 2.1 (airborne measurements)

Smooths and unifies the raw Polar 6 GPS/INS position and attitude data into a single `mastertracks_unified.nc`. The output provides the time–latitude–longitude–altitude reference onto which all other data products (T-Bird, MODIS, ERA5, CARRA) are collocated.


In [ ]:
import xarray as xr
import pandas as pd
import glob

files = glob.glob('../../../data/observations/raw/polar6/mastertracks/*.tab')
files.sort()
total_flight_hours = 0

list_of_ds = []
for f in files:
    df = pd.read_csv(f, sep='\t', encoding='latin1', skiprows=17, parse_dates=['Date/Time']).rename(columns={'Date/Time': 'time'})
    df = df.set_index('time')
    # calculate the time length of the lfigh and print it 
    flight_time = df.index[-1] - df.index[0]
    print(f'Flight time for {f.split("/")[-1].split("_mastertrack")[0]}: {flight_time}')
    total_flight_hours += flight_time.total_seconds() / 3600
    list_of_ds.append(df.to_xarray())

ds = xr.concat(list_of_ds, dim='time').sortby('time')
ds = ds.rename(
    {'Latitude' : 'latitude', 'Longitude': 'longitude', 'Flight altitude [m]': 'altitude'}
)

print('TOTAL FLIGHT HOURS: ', f'{total_flight_hours:.2f}')

ds.to_netcdf('../../../data/observations/processed/polar6_unified/mastertracks_unified.nc')

In [ ]:
total_flight_hours